# Multi-Domain Cell Representation Classification

This notebook demonstrates the core workflow for training a classifier on cell representations from multiple biological tissues/datasets and evaluating it on a target test tissue. The pipeline includes:

1. **Data Loading:** Generating simulated cell representations from multiple source domains
2. **Model Initialization:** Creating VREx classifier model
3. **Training:** Simultaneous optimization across source domains
4. **Evaluation:** Assessing performance on target test domain
5. **Result Saving:** Storing predictions and metrics


In [ ]:
# Cell 1: Imports and Setup
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Simulated VREx classifier implementation
class VREx(torch.nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        # Classifier network architecture
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(feature_dim, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
            torch.nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.classifier(x)
    
    def update(self, minibatches, optimizer, scheduler=None):
        optimizer.zero_grad()
        total_loss = 0
        
        # VREx core: Compute loss for each domain simultaneously
        for data in minibatches:
            x, y = data
            pred = self(x).squeeze()
            loss = torch.nn.functional.binary_cross_entropy(pred, y)
            total_loss += loss
        
        # Backpropagate and update weights
        total_loss.backward()
        optimizer.step()
        
        # Learning rate scheduling (optional)
        if scheduler:
            scheduler.step()
            
        return total_loss.item()

In [ ]:
# Cell 2: Simulated Data Loading
# Simulates cell representations from multiple biological datasets/tissues

# Configuration parameters
feature_dim = 100  # Dimensionality of cell representations
num_domains = 3    # Number of source domains for training

def generate_simulated_data(num_samples, feature_dim):
    """Generates synthetic cell representation data with labels"""
    # Cell representations (random normal distribution)
    X = torch.randn(num_samples, feature_dim)
    
    # Binary labels (simulated classification task)
    y = torch.randint(0, 2, (num_samples,)).float()
    
    return TensorDataset(X, y)

# Create domain datasets
train_domains = {}
for i in range(num_domains):
    domain_name = f"SourceTissue_{i+1}"
    dataset = generate_simulated_data(500, feature_dim)
    train_domains[domain_name] = DataLoader(dataset, batch_size=64, shuffle=True)

# Test domain - simulates the target tissue for evaluation
test_domain = "TargetTissue"
val_loader = DataLoader(generate_simulated_data(200, feature_dim), batch_size=64)

print(f"Training domains: {list(train_domains.keys())}")
print(f"Test domain: {test_domain}")
print(f"Feature dimensions: {feature_dim}")

In [ ]:
# Cell 3: Model Initialization and Training Setup

# Initialize VREx classifier model
model = VREx(feature_dim)
print("Model architecture:")
print(model)

# Adam optimizer configuration
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Create output directory for results
os.makedirs(f"output/{test_domain}", exist_ok=True)
print(f"\nResults will be saved to: output/{test_domain}")

In [ ]:
# Cell 4: Training Loop with Cross-Domain Validation
# Trains simultaneously on multiple source tissues/datasets
# Validates performance on target tissue after each epoch

max_epochs = 10
best_auroc = 0
best_preds = None

for epoch in range(max_epochs):
    # Training phase ------------------------------------------------------
    model.train()
    epoch_loss = 0
    batch_count = 0
    
    # Iterate through aligned batches from all source domains
    for batches in zip(*train_domains.values()):
        # Update model weights using batch from each domain
        loss = model.update(batches, optimizer)
        epoch_loss += loss
        batch_count += 1
    
    # Validation phase ----------------------------------------------------
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X, y in val_loader:
            preds = model(X).squeeze()
            y_true.extend(y.numpy())
            y_pred.extend(preds.numpy())
    
    # Calculate performance metrics
    from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
    
    auroc = roc_auc_score(y_true, y_pred)
    acc = accuracy_score(y_true, [1 if p > 0.5 else 0 for p in y_pred])
    auprc = average_precision_score(y_true, y_pred)
    
    # Print epoch summary
    print(f"Epoch {epoch+1}/{max_epochs}")
    print(f"  Training Loss: {epoch_loss/batch_count:.4f}")
    print(f"  Test AUROC:    {auroc:.4f}")
    print(f"  Test Accuracy: {acc:.4f}")
    print(f"  Test AUPRC:    {auprc:.4f}")
    
    # Save best model weights
    if auroc > best_auroc:
        best_auroc = auroc
        best_acc = acc
        best_auprc = auprc
        best_preds = y_pred
        torch.save(model.state_dict(), f"output/{test_domain}/best_model.pth")
        print("  ★ New best model saved!")


In [ ]:
# Cell 5: Final Evaluation and Result Visualization
# Saves predictions and generates performance plots

# Save predictions to CSV
results_df = pd.DataFrame({
    'true_label': y_true,
    'predicted_prob': best_preds
})
results_df.to_csv(f"output/{test_domain}/predictions.csv", index=False)

# Generate performance report
final_metrics = f"""Final Performance Report ({test_domain})
-------------------------
Accuracy:  {best_acc:.4f}
AUROC:     {best_auroc:.4f}
AUPRC:     {best_auprc:.4f}"""

with open(f"output/{test_domain}/performance_report.txt", 'w') as f:
    f.write(final_metrics)

print("\n" + final_metrics)

# Plot AUROC curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_true, best_preds)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', 
         label=f'ROC curve (AUC = {best_auroc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'Receiver Operating Characteristic ({test_domain})')
plt.legend(loc="lower right")
plt.savefig(f"output/{test_domain}/roc_curve.png", dpi=300)
plt.show()